# Processing CESM2-LENS HIST-SSP370 GHG single forcing ensemble member data

### Set up
#### Packages

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from Processing_functions import FixLongitude, FixTime, FixGrid, InterPlevels
xr.set_options(display_expand_data=False);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import random
import os

#### Filepaths & name variables

In [ ]:
## File name
cesm2LE = 'b.e21.B1850cmip6.f09_g17.'
cesm2ghg_histname = cesm2LE+'CESM2-SF-GHG'
cesm2ghg_sspname = cesm2ghg_histname+'-SSP370'
cesm2LE_outname = cesm2LE+'SF2-GHG'


## Filepaths
comp = 'atm'
freq = 0 # 0: monthly, 1: daily
var_ind = 7
path_to_arch = "/glade/campaign/collections/gdex/data/d651055/CESM2-SF/"+comp+"/proc/tseries/month_1/"

# ATM
# 0,     1     
# TREFHT, RESTOM

# ICE
# 0,    1 
# aice, hi

# OCN
# 0
# MOC

# Variables
var_list = {'atm': ['TREFHT','RESTOM'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

# Extensions
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
mod_com = {'atm': 'cam',
           'ice': 'cice',
           'ocn': 'pop'}
time_path = {'atm': ['month_1'],
                'ice': ['month_1','day_1'],
                'ocn': ['month_1']}
vert_lev = {'atm': [False,False],
            'ice': [False,False],
            'ocn': [False]}
yr_extn = {'out': [".195001-202412."]}

path_to_outdata = '/glade/work/glydia/processed_CESM2_LENS_data/longer_recent/'
plot_levels = [300,500,850,925]

In [ ]:
cluster = PBSCluster(cores    = 1,
                     memory   = '25GiB',
                     queue    = 'casper',
                     walltime = '04:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='HIST-SSP370_SFGHG_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [ ]:
client

In [ ]:
## Chunking variables
time_ch = 365*2 if freq == 1 else 600
chunks = {
    'atm': {'time': time_ch, 'lat': 96, 'lon': 144, 'lev': -1},
    'ice': {'time': time_ch, 'nj': 192, 'ni': 160},
    'ocn': {'time': time_ch, 'nlat': 64, 'nlon': 96, 'z_t': 5}
}

### Load & modify data
#### Control data

In [ ]:
ens_codes = np.array([str(i).zfill(3) for i in range(1,16)])

startyrs_hist = np.array([1950,2000])
endyrs_hist = np.array([1999,2014])

yrs_hist = []
for i in range(0,len(startyrs_hist)):
    yrs_hist.append('.'+str(startyrs_hist[i])+'01-'+str(endyrs_hist[i])+'12.')

startyrs_ssp = np.array([2015])
endyrs_ssp = np.array([2050])
yrs_ssp = ['.'+str(startyrs_ssp[0])+'01-'+str(endyrs_ssp[0])+'12.']

yr_range = np.arange(1950,2026)
yr_range = [str(i) for i in yr_range]

In [ ]:
not_vert = not vert_lev[comp][var_ind]

In [ ]:
slice_numbers = np.arange(1,16)
# slice_numbers = np.arange(1,6)

In [ ]:
%%time
if var != 'RESTOM' and not_vert:
    num = 0
    for e_code in ens_codes:
        path_list = [path_to_arch+var+'/'+cesm2ghg_histname+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr+'nc' for yr in yrs_hist]
        path_list.append(path_to_arch+var+'/'+cesm2ghg_sspname+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yrs_ssp[0]+'nc')

        print(path_list[0])

        ds = xr.open_mfdataset(path_list, chunks=chunks[comp], parallel=True)

        dsv = ds[var]
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr_sl = yr_range[i]
                endyr_sl = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
                
                fixedtime_data = FixTime(ann_slice)
                
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                elif comp == 'ocn':
                    processed_list.append(fixedtime_data)
        
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
        
        if freq == 0 and not_vert:
            processed_comp = dask.compute(*processed_list)
            print('computed list')
            
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)
                                    +h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

In [ ]:
%%time
if var != 'RESTOM' and vert_lev[comp][var_ind]:
    num = 0
    for e_code in ens_codes:
        path_list = [path_to_arch+var+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr+'nc' for yr in yrs_hist]
        path_list.append(path_to_arch+var+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+var+yrs_ssp[0]+'nc')

        print(path_list[0])

        ds = xr.open_mfdataset(path_list, chunks=chunks[comp], parallel=True)
        dsv = ds

        path_listP = [path_to_arch+'PS'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr+'nc' for yr in yrs_hist]
        path_listP.append(path_to_arch+'PS'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yrs_ssp[0]+'nc')
        dsp = xr.open_mfdataset(path_listP,chunks=chunks[comp])
            
        dsv['PS'] = dsp['PS']
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # Selection of final time
            final_time = pd.date_range(str(1950+i)+'-01-01',str(1950+i)+'-12-01',freq='MS')
                
            # Selection of data    
            startyr_sl = yr_range[i]
            endyr_sl = yr_range[i+1]
            ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
            print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
            
            ann_slice = FixTime(ann_slice)
            ann_slice = ann_slice.assign_coords(time=final_time)
            print('   fixed time')

            ann_slice = FixLongitude(ann_slice, False)
            print('   fixed longitude')
            
            ann_slice = InterPlevels(ann_slice, var)
            print('   interpolated p-levels')
            processed_list.append(ann_slice)

        
        processed_out = xr.concat(processed_list,dim='time').chunk({'time':111})
        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        slice_list.append(processed_out)  
        print('concatenated slice '+str(num))
        num = num+1
        
        for i in range(0,len(slice_list)):
            if i == 0:
                slice_list[i].to_zarr(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                                group=var)
                print('saved initial zarr store')
            else:
                print('saving slice '+str(i+1))
                slice_list[i].to_zarr(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr', 
                                        append_dim='slice', mode='a-',group=var)
        print('wrote data to disk')
    
       

In [ ]:
%%time

if var == 'RESTOM':
    num = 0
    for e_code in ens_codes:
        
        path_list_fsnt = [path_to_arch+'FSNT'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr+'nc' for yr in yrs_hist]
        path_list_fsnt.append(path_to_arch+'FSNT'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yrs_ssp[0]+'nc')

        path_list_flnt = [path_to_arch+'FLNT'+'/'+cesm2LE_HIST+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr+'nc' for yr in yrs_hist]
        path_list_flnt.append(path_to_arch+'FLNT'+'/'+cesm2LE_SSP+'.'+e_code+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yrs_ssp[0]+'nc')

        print(path_list_fsnt[0])

        ds_fsnt = xr.open_mfdataset(path_list_fsnt, chunks=chunks[comp], parallel=True)
        ds_flnt = xr.open_mfdataset(path_list_flnt, chunks=chunks[comp], parallel=True)
    
        ds_fsnt = ds_fsnt['FSNT']
        ds_flnt = ds_flnt['FLNT']

        dsv = ds_fsnt-ds_flnt
        dsv = dsv.rename(var) 
        
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr = yr_range[i]
                endyr = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr+'-02-01',endyr+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr+'-02-01 to '+endyr+'-01-17')
                
                fixedtime_data = dask.delayed(FixTime)(ann_slice)
                print('   fixed time')
        
                fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                processed_list.append(fixedgrid_data)
                print('   fixed longitude')
        
        
        processed_comp = dask.compute(*processed_list)
        print('computed list')
        
        processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
        print('concatenated data')

        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        
        processed_out.to_netcdf(path_to_outdata+cesm2LE_outname+'.'+str(slice_numbers[num]).zfill(3)+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                    format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
        print('wrote data to disk')
    
        num = num+1

In [ ]:
client.shutdown()